In [7]:
#from langchain_anthropic import ChatAnthropic
from langchain_openai import ChatOpenAI
import os
from load_dotenv import load_dotenv

load_dotenv() #This function will load everything which is saved in our .env file and will make them available
              # in the os.environ dictionary (env variables)

if os.environ.get("OPENAI_API_KEY"):
    print("Bro, API KEY Variable Exists")
else: 
    raise ValueError("OPENAI_API_KEY not found")

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser

llm_openai = ChatOpenAI(model ='gpt-5-mini', temperature=0)

Bro, API KEY Variable Exists


In [8]:
from pydantic import BaseModel
from typing import Literal

class llm_schema(BaseModel):
    movie_summary_flag: Literal["positive", "negative"] #Literal means you only have this 2 options to give the answer. Postive or Negative

llm_structured_output = llm_openai.with_structured_output(llm_schema)

#llm_structured_output.invoke("The Movie was good")

In [13]:
result = llm_structured_output.invoke("The Movie was good")
result

llm_schema(movie_summary_flag='positive')

In [15]:
result.model_dump()

{'movie_summary_flag': 'positive'}

In [16]:
result.model_dump()['movie_summary_flag']

'positive'

Now we will use above schema to build the LLM

# **CHAIN WITH Conditional Chains**

In [10]:
# TASK1 - Prompt

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a movie review evaluator"),
    ("human", "Please categorize the movie as positive or negative : {input}")])

In [11]:
# TASK2 - LLM

llm_structured_output = llm_openai.with_structured_output(llm_schema)

In [25]:
# TASK3 - Custom Runnable
from langchain_core.runnables import RunnableLambda

def pydantic_json(input:llm_schema) -> str:

    return input.model_dump()['movie_summary_flag']

pydantic_json_lambda = RunnableLambda(pydantic_json)  # Wraps function into runnable (so it can be part of chain)

Why we convert the ouput -> Dictionary, because next chains expect input like: {"text": "summary here"}

# **Conditional Chain 1**

In [17]:
# Task1- Prompt

linkedin_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a LinkedIn post generator"),
    ("human", "Create a post for the following text for LinkedIn: {text}")
])

# Task2 - LLM

llm_openai = ChatOpenAI(model ='gpt-5-mini', temperature=0)

# Task3 - Str Parser

str_parser = StrOutputParser()  

chain_linkedin = linkedin_prompt | llm_openai | str_parser

# **Conditional Chain 2**

In [21]:
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnableBranch

In [19]:
def insta_chain(text:dict):

    text = text["text"]  # Extract actual text

    insta_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a Instagram post generator"),
    ("human", "Create a post for the following text for Instagram: {text}")])

    # Task2 - LLM
    llm_openai = ChatOpenAI(model ='gpt-5-mini', temperature=0)

    # Task3 - Str Parser
    str_parser = StrOutputParser()  

    chain_insta = insta_prompt | llm_openai | str_parser

    result = chain_insta.invoke(text)

    return result

insta_chain_runnable = RunnableLambda(insta_chain)  #creating a lambda function for it.

## **Final Orchestration**

In [26]:
conditional_chain = RunnableBranch(
    (lambda x: "positive" in x, chain_linkedin),
    insta_chain_runnable
)

final_orchestrator = prompt_template | llm_structured_output | pydantic_json_lambda | conditional_chain

In [27]:
final_orchestrator.invoke({"input": "I love this KGF movie"})

"Positive isn't just a mood — it's a strategy.\n\nChoosing a positive mindset helps teams stay resilient, turn setbacks into learning moments, and keeps focus on solutions instead of obstacles. When we lead with optimism, we create space for creativity, collaboration, and steady progress.\n\nSmall habits make a big difference: acknowledge wins, reframe challenges, and encourage others. What’s one thing you do to stay positive at work? Share it below — let’s learn from each other.\n\n#Positive #Mindset #Leadership #Growth #WorkplaceWellbeing"